# Spatial search: random walk vs Szegedy walk

Search a graph for marked vertices. Classical random walk is O(N); Szegedy walk is O(√N) — the canonical Grover-style quadratic speedup.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**Problem:** spatial search on a graph. **Classical:** random walk hitting time O(N). **Quantum:** Szegedy walk, amplitude amplification, hitting time O(√N).

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.domains.optimization.kernels import (
    markov_chain_transition, quantum_walk_operator,
)
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

ADJ = np.array([
    [0., 1., 1., 0.],
    [1., 0., 1., 1.],
    [1., 1., 0., 1.],
    [0., 1., 1., 0.],
])
N = 4
nn = N * N

@trace
def prog(adj, state):
    p_flat = markov_chain_transition(adj, n_nodes=N)
    u_flat = quantum_walk_operator(p_flat, n_nodes=N)
    u_mat = shape.reshape(u_flat, result_type=types.tensor("f64", [nn, nn]),
                          shape=[nn, nn])
    return apply_linear(u_mat, state)

state = np.ones(nn) / np.sqrt(nn)
module = prog(ADJ.tolist(), state.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
